# Coastal flood avoided EAD and mangrove benefit locations

Exploratory two-panel map equivalent to the river-flood restoration-location figure. Panel (a) maps infrastructure assets with positive avoided coastal-flood EAD under the maximum scenario. Panel (b) maps mangrove patches attributed with positive avoided coastal-flood EAD under the final weighted area-distance attribution.

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing dphil_papers")


ROOT = find_project_root()
PAPERS = ROOT / "dphil_papers"
ROBYN_LIBRARIES = PAPERS / "robyns_libraries"
if str(ROBYN_LIBRARIES) not in sys.path:
    sys.path.append(str(ROBYN_LIBRARIES))

import Robyn_paper_2_defs

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
PAPER3 = PAPERS / "dphil_paper_3"
COMMON = PAPERS / "dphil_common_cross_cutting"

SCENARIO = "maximum"
CRS_METRIC = "EPSG:3448"
ASSET_COLOR_CAP_QUANTILE = 0.995
MANGROVE_COLOR_CAP_QUANTILE = 0.995

BOUNDARY_PATH = COMMON / "common_incoming_data" / "boundaries" / "jam_adm_shp" / "jam_admbnda_adm0.shp"
ASSET_EAD_PATH = PAPER3 / "results" / "02_damage_estimates" / "coastal_flood_damages" / "results_coastal_maximum_scenario" / "damage_estimates" / "coastal_ead_asset_level_usd.csv"
ASSET_GEOMETRY_CACHE_PATH = PAPER3 / "results_coastal_scenario_comparison" / "maps_min_max_comparison" / "cache" / "maximum_all_sector_map_layer.geoparquet"
MANGROVE_ATTRIBUTION_PATH = PAPER3 / "results_coastal_scenario_comparison" / "weighted_area_distance_signed" / "mangrove_priority_ranking" / "mangrove_priority_map_weighted_area_distance.gpkg"

OUTPUT_DIR = PAPER3 / "results" / "03_cross_hazard_comparison" / "spatial_service_provision_comparison" / "coastal_avoided_ead_mangrove_locations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for input_path in [BOUNDARY_PATH, ASSET_EAD_PATH, ASSET_GEOMETRY_CACHE_PATH, MANGROVE_ATTRIBUTION_PATH]:
    if not input_path.exists():
        raise FileNotFoundError(input_path)

OUTPUT_DIR


## Load and prepare map layers

In [ ]:
boundary = gpd.read_file(BOUNDARY_PATH).to_crs(CRS_METRIC)
boundary_parts = boundary.explode(index_parts=False).copy()
main_island_parts = boundary_parts.loc[boundary_parts.geometry.area > 1_000_000].copy()
MAIN_ISLAND_BOUNDS = main_island_parts.total_bounds
SOUTHEAST_ZOOM_BOUNDS = np.array([700_000, 630_000, 785_000, 665_000], dtype=float)

asset_key_columns = ["Sector", "Subsector", "Asset", "Layer", "Asset_ID"]
asset_ead = pd.read_csv(ASSET_EAD_PATH)
positive_asset_ead = asset_ead.loc[asset_ead["Avoided_EAD_USD"] > 0].copy()

asset_geometry_parts = gpd.read_parquet(
    ASSET_GEOMETRY_CACHE_PATH,
    columns=[*asset_key_columns, "geometry"],
).to_crs(CRS_METRIC)
positive_asset_geometry_parts = asset_geometry_parts.merge(
    positive_asset_ead[asset_key_columns],
    on=asset_key_columns,
    how="inner",
)
positive_asset_geometries = positive_asset_geometry_parts.dissolve(
    by=asset_key_columns,
    as_index=False,
)
positive_asset_locations = positive_asset_geometries.merge(
    positive_asset_ead,
    on=asset_key_columns,
    how="left",
)

missing_asset_geometry = positive_asset_ead.merge(
    positive_asset_geometries[asset_key_columns],
    on=asset_key_columns,
    how="left",
    indicator=True,
).loc[lambda dataframe: dataframe["_merge"].eq("left_only")]

mangrove_attribution = gpd.read_file(MANGROVE_ATTRIBUTION_PATH).to_crs(CRS_METRIC)
positive_mangrove_attribution = mangrove_attribution.loc[mangrove_attribution["avoided_usd_max"] > 0].copy()

positive_asset_locations["plot_avoided_ead_usd"] = positive_asset_locations["Avoided_EAD_USD"].clip(
    upper=positive_asset_locations["Avoided_EAD_USD"].quantile(ASSET_COLOR_CAP_QUANTILE)
)
positive_mangrove_attribution["plot_avoided_ead_thousand_usd"] = (
    positive_mangrove_attribution["avoided_usd_max"] / 1_000
).clip(
    upper=(positive_mangrove_attribution["avoided_usd_max"] / 1_000).quantile(MANGROVE_COLOR_CAP_QUANTILE)
)

summary = pd.DataFrame(
    [
        {
            "layer": "positive_asset_locations",
            "feature_count": len(positive_asset_locations),
            "total_positive_avoided_ead_usd": positive_asset_locations["Avoided_EAD_USD"].sum(),
            "colour_cap_quantile": ASSET_COLOR_CAP_QUANTILE,
            "colour_cap_value": positive_asset_locations["plot_avoided_ead_usd"].max(),
            "missing_geometry_count": len(missing_asset_geometry),
        },
        {
            "layer": "positive_mangrove_patches",
            "feature_count": len(positive_mangrove_attribution),
            "total_positive_avoided_ead_usd": positive_mangrove_attribution["avoided_usd_max"].sum(),
            "colour_cap_quantile": MANGROVE_COLOR_CAP_QUANTILE,
            "colour_cap_value": positive_mangrove_attribution["plot_avoided_ead_thousand_usd"].max(),
            "missing_geometry_count": 0,
        },
    ]
)

summary


In [ ]:
positive_asset_locations_path = OUTPUT_DIR / "coastal_positive_avoided_asset_locations_maximum.geoparquet"
positive_mangrove_locations_path = OUTPUT_DIR / "coastal_positive_avoided_mangrove_locations_maximum.gpkg"
summary_path = OUTPUT_DIR / "coastal_avoided_ead_mangrove_locations_summary.csv"

positive_asset_locations.to_parquet(positive_asset_locations_path, index=False)
positive_mangrove_attribution.to_file(positive_mangrove_locations_path, driver="GPKG")
summary.to_csv(summary_path, index=False)

positive_asset_locations_path, positive_mangrove_locations_path, summary_path


## Draw coastal equivalent of river-flood location figure

In [ ]:
NATURE_RC = {
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "figure.titlesize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
}

ASSET_CMAP = mpl.colormaps["magma_r"]
MANGROVE_CMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "mangrove_green_blue_black",
    ["#66bd63", "#1a9850", "#2b8cbe", "#08519c", "#000000"],
)
BOUNDARY_COLOR = "#4d4d4d"
NO_BENEFIT_MANGROVE_COLOR = "#f1f1f1"
POSITIVE_MANGROVE_EDGE_COLOR = "#005a32"


def format_number_tick(value: float) -> str:
    if value >= 1_000:
        return f"{value:,.0f}"
    if value >= 10:
        return f"{value:.0f}"
    return f"{value:.1f}"


def set_map_extent(axis, map_bounds, right_padding_fraction: float = 0.18) -> None:
    x_min, y_min, x_max, y_max = map_bounds
    x_range = x_max - x_min
    y_range = y_max - y_min
    axis.set_xlim(x_min - 0.02 * x_range, x_max + right_padding_fraction * x_range)
    axis.set_ylim(y_min - 0.05 * y_range, y_max + 0.08 * y_range)


def add_standard_map_furniture(axis) -> None:
    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.86, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(
        axis,
        location=(0.86, 0.86),
        size=0.05,
        fontsize=8,
        label_offset=0.02,
    )


def plot_geometry_types(axis, geodataframe: gpd.GeoDataFrame, value_column: str, cmap, norm: Normalize) -> None:
    polygon_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    line_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    point_geometries = geodataframe.loc[geodataframe.geometry.geom_type.isin(["Point", "MultiPoint"])].copy()

    if len(polygon_geometries) > 0:
        polygon_geometries.plot(
            ax=axis,
            column=value_column,
            cmap=cmap,
            norm=norm,
            linewidth=0,
            alpha=0.95,
            zorder=2,
        )
    if len(line_geometries) > 0:
        line_geometries.plot(
            ax=axis,
            column=value_column,
            cmap=cmap,
            norm=norm,
            linewidth=0.45,
            alpha=0.95,
            zorder=3,
        )
    if len(point_geometries) > 0:
        point_geometries.plot(
            ax=axis,
            column=value_column,
            cmap=cmap,
            norm=norm,
            markersize=5,
            alpha=0.95,
            zorder=4,
        )


def add_colorbar(figure, axis, cmap, norm: Normalize, label: str, top_label_prefix: str | None = None) -> None:
    divider = make_axes_locatable(axis)
    colorbar_axis = divider.append_axes("right", size="2.8%", pad=0.04)
    scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])
    colorbar = figure.colorbar(scalar_mappable, cax=colorbar_axis)
    colorbar.outline.set_linewidth(0.35)
    colorbar.ax.tick_params(width=0.35, length=2, labelsize=6)
    colorbar.set_label(label, fontsize=6.5)
    tick_values = colorbar.get_ticks()
    tick_labels = [format_number_tick(tick_value) for tick_value in tick_values]
    if top_label_prefix is not None and len(tick_labels) > 0:
        tick_labels[-1] = f"{top_label_prefix}{tick_labels[-1]}"
    colorbar.set_ticklabels(tick_labels)


def draw_coastal_location_panel(output_stem: str, map_bounds=MAIN_ISLAND_BOUNDS, right_padding_fraction: float = 0.18, dpi: int = 600):
    asset_norm = Normalize(
        vmin=0,
        vmax=float(positive_asset_locations["plot_avoided_ead_usd"].max()),
    )
    mangrove_norm = Normalize(
        vmin=0,
        vmax=float(positive_mangrove_attribution["plot_avoided_ead_thousand_usd"].max()),
    )

    with mpl.rc_context(NATURE_RC):
        figure, axes = plt.subplots(2, 1, figsize=(7.1, 6.2), constrained_layout=False)
        plt.subplots_adjust(left=0.02, right=0.92, top=0.97, bottom=0.04, hspace=0.08)

        asset_axis = axes[0]
        asset_axis.set_axis_off()
        asset_axis.set_aspect("equal")
        main_island_parts.boundary.plot(ax=asset_axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=5)
        plot_geometry_types(asset_axis, positive_asset_locations, "plot_avoided_ead_usd", ASSET_CMAP, asset_norm)
        main_island_parts.boundary.plot(ax=asset_axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
        set_map_extent(asset_axis, map_bounds, right_padding_fraction)
        add_standard_map_furniture(asset_axis)
        asset_axis.text(0.01, 0.98, "(a)", transform=asset_axis.transAxes, ha="left", va="top", fontsize=9, fontweight="bold")
        asset_axis.set_title("Infrastructure locations with positive avoided coastal-flood EAD", loc="left", pad=2)
        add_colorbar(
            figure,
            asset_axis,
            ASSET_CMAP,
            asset_norm,
            "Avoided EAD (US$)",
            top_label_prefix=None,
        )

        mangrove_axis = axes[1]
        mangrove_axis.set_axis_off()
        mangrove_axis.set_aspect("equal")
        main_island_parts.boundary.plot(ax=mangrove_axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=5)
        mangrove_attribution.plot(
            ax=mangrove_axis,
            color=NO_BENEFIT_MANGROVE_COLOR,
            edgecolor="none",
            zorder=2,
        )
        positive_mangrove_attribution.plot(
            ax=mangrove_axis,
            column="plot_avoided_ead_thousand_usd",
            cmap=MANGROVE_CMAP,
            norm=mangrove_norm,
            linewidth=0,
            alpha=0.98,
            zorder=3,
        )
        positive_mangrove_attribution.boundary.plot(
            ax=mangrove_axis,
            color=POSITIVE_MANGROVE_EDGE_COLOR,
            linewidth=0.18,
            alpha=0.95,
            zorder=4,
        )
        main_island_parts.boundary.plot(ax=mangrove_axis, color=BOUNDARY_COLOR, linewidth=0.65, zorder=6)
        set_map_extent(mangrove_axis, map_bounds, right_padding_fraction)
        add_standard_map_furniture(mangrove_axis)
        mangrove_axis.text(0.01, 0.98, "(b)", transform=mangrove_axis.transAxes, ha="left", va="top", fontsize=9, fontweight="bold")
        mangrove_axis.set_title("Mangrove patches attributed with positive avoided coastal-flood EAD", loc="left", pad=2)
        add_colorbar(
            figure,
            mangrove_axis,
            MANGROVE_CMAP,
            mangrove_norm,
            "Attributed avoided EAD (US$ thousand)",
            top_label_prefix=None,
        )

        output_paths = []
        for suffix in ["png", "pdf", "svg"]:
            output_path = OUTPUT_DIR / f"{output_stem}.{suffix}"
            save_kwargs = {"bbox_inches": "tight", "facecolor": "white"}
            if suffix == "png":
                save_kwargs["dpi"] = dpi
            figure.savefig(output_path, **save_kwargs)
            output_paths.append(output_path)
        plt.show()
        return output_paths


In [ ]:
base_national_output_paths = draw_coastal_location_panel(
    "coastal_avoided_ead_asset_and_mangrove_benefit_locations_maximum"
)
national_output_paths = draw_coastal_location_panel(
    "coastal_avoided_ead_asset_and_mangrove_benefit_locations_maximum_national"
)
southeast_zoom_output_paths = draw_coastal_location_panel(
    "coastal_avoided_ead_asset_and_mangrove_benefit_locations_maximum_southeast_zoom",
    map_bounds=SOUTHEAST_ZOOM_BOUNDS,
    right_padding_fraction=0.10,
)
output_paths = base_national_output_paths + national_output_paths + southeast_zoom_output_paths
output_paths


In [ ]:
print("Wrote figure outputs to:", OUTPUT_DIR)
for output_path in output_paths:
    print(" -", output_path.name)
print("Wrote data outputs:")
for output_path in [positive_asset_locations_path, positive_mangrove_locations_path, summary_path]:
    print(" -", output_path.name)
summary
